![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [1]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging, pipeline
logging.set_verbosity(logging.WARNING)

In [2]:
import evaluate

Classify the sentiment of the five car reviews in the `car_reviews.csv`

In [3]:
df = pd.read_csv('data/car_reviews.csv', sep=';')

In [4]:
X = df['Review'].tolist()
y = df['Class'].tolist()

In [5]:
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')
predicted_labels = classifier(X)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [6]:
predictions = [1 if result['label'] == 'POSITIVE' else 0 for result in predicted_labels]
ref = [1 if label == 'POSITIVE' else 0 for label in y]

In [7]:
acc = evaluate.load('accuracy')
f1 = evaluate.load('f1')

accuracy_result = acc.compute(predictions=predictions, references=ref)
f1_result = f1.compute(predictions=predictions, references=ref)

print(f"[+] Accuracy Telemetry: {accuracy_result}")
print(f"[+] F1 Score Telemetry: {f1_result}")

[+] Accuracy Telemetry: {'accuracy': 0.8}
[+] F1 Score Telemetry: {'f1': 0.8571428571428571}


Pass the first two sentences of the first review in the dataset to an English-to-Spanish translation

In [8]:
first_review = df['Review'].iloc[0]
print(first_review)

I am very satisfied with my 2014 Nissan NV SL. I use this van for my business deliveries and personal use. Camping, road trips, etc. We dont have any children so I store most of the seats in my warehouse. I wanted the passenger van for the rear air conditioning. We drove our van from Florida to California for a Cross Country trip in 2014. We averaged about 18 mpg. We drove thru a lot of rain and It was a very comfortable and stable vehicle. The V8 Nissan Titan engine is a 500k mile engine. It has been tested many times by delivery and trucking companies. This is why Nissan gives you a 5 year or 100k mile bumper to bumper warranty. Many people are scared about driving this van because of its size. But with front and rear sonar sensors, large mirrors and the back up camera. It is easy to drive. The front and rear sensors also monitor the front and rear sides of the bumpers making it easier to park close to objects. Our Nissan NV is a Tow Monster. It pulls our 5000 pound travel trailer li

In [9]:
sentences = first_review.split('.')[:2]
sentences = '.'.join(sentences)

In [10]:
translator = pipeline('translation_en_to_es', model='Helsinki-NLP/opus-mt-en-es')

translated_review = translator(sentences)[0]['translation_text']
print(translated_review)

Device set to use cpu


Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal


In [11]:
with open('data/reference_translations.txt', 'r') as f:
    references = [f.read().strip()]

In [12]:
references

['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.\nEstoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']

In [13]:
bleu = evaluate.load('bleu')
bleu_score = bleu.compute(predictions=[translated_review], references=references)['bleu']

In [14]:
bleu_score

0.2822068036100631

Extractive QA LLM

In [15]:
context = df['Review'].iloc[1]
print(context)

The car is fine. It's a bit loud and not very powerful. On one hand, compared to its peers, the interior is well-built. The transmission failed a few years ago, and the dealer replaced it under warranty with no issues. Now, about 60k miles later, the transmission is failing again. It sounds like a truck, and the issues are well-documented. The dealer tells me it is normal, refusing to do anything to resolve the issue. After owning the car for 4 years, there are many other vehicles I would purchase over this one. Initially, I really liked what the brand is about: ride quality, reliability, etc. But I will not purchase another one. Despite these concerns, I must say, the level of comfort in the car has always been satisfactory, but not worth the rest of issues found.


In [16]:
qa_extractor = pipeline(task='question-answering',model='deepset/minilm-uncased-squad2')

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


In [17]:
question = "What did he like about the brand?"

In [18]:
answer = qa_extractor(question=question, context=context)['answer']

In [19]:
print(answer)

ride quality, reliability


Summarize the last review in the dataset, into approximately 50-55 tokens long

In [20]:
last_review = df['Review'].iloc[-1]

In [21]:
summarizer = pipeline(task='summarization', model='facebook/bart-large-cnn')

Device set to use cpu


In [22]:
summary_output = summarizer(last_review, min_length=50, max_length=55, do_sample=False)[0]
summarized_text = summary_output['summary_text']

In [23]:
print(last_review)

I've been dreaming of owning an SUV for quite a while, but I've been driving cars that were already paid for during an extended period. I ultimately made the decision to transition to a brand-new car, which, of course, involved taking on new payments. However, given that I don't drive extensively, I was inclined to avoid a substantial financial commitment. The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment; the financial arrangement is quite reasonable. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. I am VERY satisfied overall. I find myself needing to exercise extra caution when making lane changes, particularly owing to the blind spots resulting from the small side windows situated towards the rear of the vehicle. To address this concern, I am actively engaged in making adjustments to my mirrors and consciously reducing the frequency of lane changes. The

In [24]:
print(summarized_text)

The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong
